# Task 2 — The Mystery Black Box
### Testing: Linear Transformations

## The scenario

Somewhere in `src/secret_box.py` there's a function `black_box(point)`.
It takes a 2D point `(x, y)` and returns a transformed point. Internally, it
applies some **composition of standard linear transformations** (rotation,
scaling, reflection, shear) — but you are not supposed to open that file and
read the source. Your job is to figure out *exactly* what the combined
transformation matrix is, using nothing but carefully chosen probe inputs
and linear algebra.

This mirrors a real situation: reverse-engineering an unknown image
transform, camera calibration matrix, or a layer of a neural network acting
on coordinates, purely from input/output pairs — no access to the internals.

## Step 0 — Why this is even possible

A **linear transformation** `T` on R^2 is completely determined by what it
does to the two standard basis vectors:

```
e1 = (1, 0)
e2 = (0, 1)
```

Why? Because *any* point `(x, y)` can be written as `x*e1 + y*e2`, and
linearity means:

```
T(x, y) = T(x*e1 + y*e2) = x*T(e1) + y*T(e2)
```

So if we know `T(e1)` and `T(e2)`, we know `T` completely. In matrix terms:
**the columns of the transformation matrix are exactly `T(e1)` and
`T(e2)`.** This is the single most important fact in this notebook —
everything else is just using it.

In [ ]:
import sys
sys.path.insert(0, "../src")
from linalg import matmul, mat_vec_mul, apply_transformation, rotation_matrix, scaling_matrix, shear_matrix, reflection_matrix, compose
import matplotlib.pyplot as plt

from secret_box import black_box  # the only thing we're "allowed" to call


## Step 1 — Probe with the standard basis vectors

We feed `e1 = (1, 0)` and `e2 = (0, 1)` into the black box. Whatever comes
out **are the columns of the hidden matrix**, full stop.

In [ ]:
e1 = (1, 0)
e2 = (0, 1)

col1 = black_box(e1)
col2 = black_box(e2)

print("black_box(1, 0) =", col1, " <- this is column 1 of the hidden matrix")
print("black_box(0, 1) =", col2, " <- this is column 2 of the hidden matrix")

# Reassemble the matrix from these two probes
deduced_matrix = [
    [col1[0], col2[0]],
    [col1[1], col2[1]],
]
print()
print("Deduced matrix:")
for row in deduced_matrix:
    print(row)


## Step 2 — Verify the deduction

A matrix deduced from just 2 probes is only useful if it correctly predicts
*every other* input. Let's test it against the real black box on a handful
of fresh points it hasn't seen.

In [ ]:
test_points = [(2, 3), (-1, 4), (5, -2), (0.5, 0.5), (3, 3)]

print(f"{'point':>12} | {'black_box':>28} | {'deduced matrix':>28} | match?")
all_match = True
for p in test_points:
    actual = black_box(p)
    predicted = mat_vec_mul(deduced_matrix, p)
    match = all(abs(a - b) < 1e-9 for a, b in zip(actual, predicted))
    all_match &= match
    print(f"{str(p):>12} | {str(actual):>28} | {str(predicted):>28} | {match}")

print()
print("All predictions matched!" if all_match else "Mismatch found -- something's wrong.")


## Step 3 — Visualize it on a whole shape

Numbers convince the head, pictures convince the gut. Let's run a simple
house-shaped set of points through both the black box and our deduced
matrix, and overlay the results.

In [ ]:
house = [(0, 0), (2, 0), (2, 1.5), (1, 2.5), (0, 1.5), (0, 0)]  # closed polygon

house_via_blackbox = [black_box(p) for p in house]
house_via_deduced = apply_transformation(deduced_matrix, house)

fig, ax = plt.subplots(figsize=(6, 6))

def plot_shape(ax, pts, style, **kwargs):
    xs, ys = zip(*pts)
    ax.plot(xs, ys, style, **kwargs)

plot_shape(ax, house, "o--", color="lightgray", label="original")
plot_shape(ax, house_via_blackbox, "o-", color="crimson", linewidth=3, label="black_box(house)")
plot_shape(ax, house_via_deduced, "x--", color="black", markersize=10, label="deduced_matrix @ house")

ax.set_aspect("equal")
ax.legend()
ax.set_title("Black box output vs. our deduced matrix (should overlap exactly)")
ax.grid(True, linewidth=0.3)
plt.show()


## Step 4 — Decompose the mystery: what elementary moves make it up?

The hint in this exercise was that the composite is built from 3 elementary
transformations: a **shear**, a **non-uniform scale**, and a **rotation**,
composed in *some* order. Composition order matters because matrix
multiplication is **not commutative** — let's prove that to ourselves, then
try to match our deduced matrix against candidate orderings.

In [ ]:
# Proof that order matters: build the same three elementary matrices two
# different ways and show the results differ.
shear = shear_matrix(kx=0.5, ky=0)
scale = scaling_matrix(sx=2, sy=1)
rotate = rotation_matrix(theta_degrees=30)

order_A = compose(rotate, scale, shear)   # Rotate( Scale( Shear(p) ) )
order_B = compose(shear, scale, rotate)   # Shear( Scale( Rotate(p) ) )

print("Rotate(Scale(Shear(x))):")
for row in order_A:
    print(["%.4f" % v for v in row])

print()
print("Shear(Scale(Rotate(x))):")
for row in order_B:
    print(["%.4f" % v for v in row])

print()
print("Are they the same matrix?", all(
    abs(order_A[i][j] - order_B[i][j]) < 1e-9
    for i in range(2) for j in range(2)
))


In [ ]:
# Now compare our deduced matrix to candidate compositions to find the
# order (and parameters) that was actually used inside the black box.
candidates = {
    "Rotate(Scale(Shear(x)))": compose(rotate, scale, shear),
    "Scale(Rotate(Shear(x)))": compose(scale, rotate, shear),
    "Shear(Scale(Rotate(x)))": compose(shear, scale, rotate),
    "Shear(Rotate(Scale(x)))": compose(shear, rotate, scale),
    "Scale(Shear(Rotate(x)))": compose(scale, shear, rotate),
    "Rotate(Shear(Scale(x)))": compose(rotate, shear, scale),
}

def matrices_close(A, B, tol=1e-6):
    return all(abs(A[i][j] - B[i][j]) < tol for i in range(2) for j in range(2))

for name, M in candidates.items():
    print(name, "-> matches deduced matrix?", matrices_close(M, deduced_matrix))
